# [ Computer Vision ] [ CSE445 ] [ Assignment-2 ]

**Dataset Name:** IoTKITs<br>
**Source Link:** https://data.mendeley.com/datasets/x5thzmkxhy/1

## Best Model Justification from `Part A` Comparison

**How we picked the best model:** we compared all three Part A YOLO models using test-split mAP@50:95, since that's what the assignment asks us to use. RF-DETR-Nano is shown below for context only — the assignment's prerequisite selection is limited to the YOLO variants, so RF-DETR isn't part of the actual pick.

| Model | mAP@50 | mAP@50:95 | Precision | Recall | F1 | FPS |
|---|---|---|---|---|---|---|
| YOLOv10s | 0.9474 | 0.9167 | 0.9646 | 0.9497 | 0.9571 | 78.83 |
| **YOLOv12s** | **0.9795** | **0.9461** | **0.9760** | **0.9805** | **0.9782** | 55.15 |
| YOLOv26s | 0.9330 | 0.9044 | 0.9464 | 0.9358 | 0.9411 | 72.20 |
| RF-DETR-Nano *(reference only, not eligible)* | 0.8601 | 0.8394 | 0.6915 | 0.9015 | 0.7671 | 33.51 |

**We picked YOLOv12s.** Among the three eligible YOLO variants, it's ahead on every single accuracy number — mAP@50:95 (0.9461, about 3 points higher than YOLOv10s and 4 points higher than YOLOv26s), plus mAP@50, precision, recall, and F1. So it was the obvious pick based on the assignment's own criterion (test mAP@50:95).

**One trade-off worth mentioning:** YOLOv12s isn't the fastest — it runs at 55.15 FPS, while YOLOv10s hits 78.83 FPS and YOLOv26s hits 72.20 FPS. So we're giving up around 30% inference speed compared to YOLOv10s. That's fine here though, because Part B is about testing how much labelled data SSL pretraining can save us — not about real-time speed. And honestly, all three models are fast enough for real-time use anyway on a T4 GPU.

**Where RF-DETR fits in:** it's clearly behind all three YOLO models on this dataset — lowest mAP@50:95 (0.8394), lowest precision (0.6915), and slowest (33.51 FPS). It was excluded from Part A's model-selection race by the assignment itself, so this isn't held against it as a failure, just noted for completeness since it's part of the same overall project.

**Bottom line:** YOLOv12s is what we're using as `PART_A_BEST_CKPT` for the rest of Part B — in the baseline notebook, and as the 100%-label reference point when comparing the SSL methods.

## Configuration

In [1]:
!pip install ultralytics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.2 MB/s eta 0:00:00


In [2]:
import json
import os
import shutil
import pandas as pd

# Paths
dataset_dir = "/kaggle/input/datasets/Mrpaul0007/iotkits"

splits_dataset_dir = "/kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition"
split_dir = os.path.join(splits_dataset_dir, "splits")

working_dir = "/kaggle/working/"
os.makedirs(working_dir, exist_ok=True)

train_dir = os.path.join(dataset_dir, "train")
valid_dir = os.path.join(dataset_dir, "valid")
test_dir  = os.path.join(dataset_dir, "test")

train_annotation = os.path.join(train_dir, "_annotations.coco.json")
valid_annotation = os.path.join(valid_dir, "_annotations.coco.json")
test_annotation  = os.path.join(test_dir, "_annotations.coco.json")

role_of_folder = {
    "train": "ssl_pool",
    "valid": "val",
    "test":  "test",
}

SEED   = 42
RHO    = 0.20
IMGSZ  = 640
BATCH  = 16
EPOCHS = 200

# Model Name (Part A)
BEST_YAML = "yolo12s.yaml"
BEST_PT   = "yolo12s.pt"

# Checkpoints
PART_A_BEST_CKPT = "/kaggle/input/notebooks/Mrpaul0007/nb-3-yolov12/checkpoints/full_train/weights/best.pt"

print("====== Configuration ======\n")
print("Dataset path       :", dataset_dir)
print("Splits dataset path:", split_dir)
print("Seed               :", SEED)
print("Label fraction rho :", RHO)
print("Image size         :", IMGSZ)
print("Batch              :", BATCH)
print("Epochs             :", EPOCHS)
print("Configuration loaded successfully!")

====== Configuration ======

Dataset path       : /kaggle/input/datasets/Mrpaul0007/iotkits
Splits dataset path: /kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition/splits
Seed               : 42
Label fraction rho : 0.2
Image size         : 640
Batch              : 16
Epochs             : 200
Configuration loaded successfully!


## Using data.yaml From `Notebook-00-Partition` Output

In [3]:
import os

old_yaml_path = os.path.join(splits_dataset_dir, "yolo_rho20", "data.yaml")

with open(old_yaml_path) as f:
    lines = f.readlines()

fixed_lines = []
for line in lines:
    if line.startswith("path:"):
        fixed_lines.append(f"path: {os.path.join(splits_dataset_dir, 'yolo_rho20')}\n")
    else:
        fixed_lines.append(line)

yaml_path = os.path.join(working_dir, "data.yaml")
with open(yaml_path, "w") as f:
    f.writelines(fixed_lines)

print("Fixed data.yaml saved at:", yaml_path)
print(open(yaml_path).read())

Fixed data.yaml saved at: /kaggle/working/data.yaml
path: /kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition/yolo_rho20
train: images/train
val: images/val
test: images/test
nc: 32
names: ["Arduino Mega 2560 -Black-", "Arduino Mega 2560 -Blue-", "Arduino Mega 2560 -Yellow-", "Arduino Uno -Black-", "Arduino Uno -Green-", "Arduino Uno Camera Shield", "Arduino Uno WiFi Shield", "Arduino-Due", "Arduino-Micro", "Arduino-Nano", "Arduino-ProMini", "Arduino-Zero", "Arduino-leonardo", "ESP32", "Esp8266", "Jetson Nano", "Jetson TX2", "Raspberry Pi 1 B-", "Raspberry Pi 3 B-", "Raspberry-Pi-1-A-", "Raspberry-Pi-2-B", "Raspberry-Pi-3-A-", "Raspberry-Pi-3-B", "Raspberry-Pi-4-B", "Raspberry-Pi-5", "Raspberry-Pi-Zero", "Raspberry-Pi-Zero-2-W", "Raspberry-Pi-Zero-W", "Raspberry-Pi-Zero-WH", "STM32", "TelosB", "Wemos"]



## Baseline(a): Random Init

In [4]:
from ultralytics import YOLO

m_rand = YOLO(BEST_YAML)
m_rand.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    pretrained=False,
    project="/kaggle/working",
    name="base_random",
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.135 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/200      6.11G      3.071      5.161      4.187         33        640: 100% ━━━━━━━━━━━━ 39/39 2.6it/s 14.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.4it/s 3.0s
                   all        311        322          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200      6.11G      3.037      5.142      4.116         35        640: 100% ━━━━━━━━━━━━ 39/39 2.6it/s 14.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.4it/s 3.0s
                   all        311        322          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/200       6.1G      2.961      5.094      3.927         42        640: 100% ━━━━━━━━━━━━ 39/39 2.6it/s 14.9s
                 Class     Images  Instances      Box(P    

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fc853a8a810>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,

## Baseline(b): COCO-pretrained

In [5]:
m_coco = YOLO(BEST_PT)
m_coco.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    project="/kaggle/working",
    name="base_coco",
)

Ultralytics 8.4.135 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=base_coco, nbs=64, nms=False, opset=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fc8bb576fc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,

## Evaluate Both Baseline

In [6]:
rows = []

for m, tag in [(m_rand, "random"), (m_coco, "coco")]:
    r = m.val(split="test", conf=0.25, iou=0.7)
    rows.append({
        "model":     tag,
        "rho":       RHO,
        "precision": r.box.mp,
        "recall":    r.box.mr,
        "map50":     r.box.map50,
        "map50_95":  r.box.map,
    })
    print(tag, r.box.mp, r.box.mr, r.box.map50, r.box.map)

Ultralytics 8.4.135 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO12s summary (fused): 159 layers, 9,243,264 parameters, 0 gradients, 23.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 4.8±1.4 MB/s, size: 40.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition/yolo_rho20/labels/test... 311 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 311/311 165.5it/s 1.9s
WARNING ⚠️ val: Cache directory /kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition/yolo_rho20/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.5it/s 5.7s
                   all        311        325      0.721      0.793      0.726      0.668
Arduino Mega 2560 -Black-    

## Part A Best Checkpoint

In [7]:
m_ref = YOLO(PART_A_BEST_CKPT)
r_ref = m_ref.val(data=yaml_path, split="test", conf=0.25, iou=0.7)

rows.append({
    "model":     "part_a_100pct_reference",
    "rho":       1.00,
    "precision": r_ref.box.mp,
    "recall":    r_ref.box.mr,
    "map50":     r_ref.box.map50,
    "map50_95":  r_ref.box.map,
})

print("part_a_100pct_reference", r_ref.box.mp, r_ref.box.mr, r_ref.box.map50, r_ref.box.map)

Ultralytics 8.4.135 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv12s summary (fused): 159 layers, 9,243,264 parameters, 0 gradients, 23.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 60.3±44.0 MB/s, size: 42.5 KB)
val: Scanning /kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition/yolo_rho20/labels/test... 311 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 311/311 918.8it/s 0.3s
WARNING ⚠️ val: Cache directory /kaggle/input/notebooks/Mrpaul0007/self-supervised-learning-nb-0-partition/yolo_rho20/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 3.4it/s 5.9s
                   all        311        325      0.976       0.98       0.98      0.946
Arduino Mega 2560 -Black-         10         10          1          1      0.995      0.995
Arduino Mega 2560 -Blue-         10         10          1          1      0.995      0.99

## Save Csv

In [8]:
df = pd.DataFrame(rows)
print(df)

csv_path = os.path.join(working_dir, "baselines.csv")
df.to_csv(csv_path, index=False)
print()
print("Saved:", csv_path)

                     model  rho  precision    recall     map50  map50_95
0                   random  0.2   0.720691  0.793334  0.725790  0.668371
1                     coco  0.2   0.907575  0.887845  0.885514  0.852544
2  part_a_100pct_reference  1.0   0.976009  0.980463  0.979531  0.946082

Saved: /kaggle/working/baselines.csv
